In [1]:
# ============================================================
# STEP 1: INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q transformers sentencepiece sacrebleu accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 6.3 MB/s eta 0:00:00


In [3]:
# ============================================================
# STEP 2: MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


In [4]:
# ============================================================
# STEP 3: LOAD NLLB ENGLISH -> HINDI MODEL
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "facebook/nllb-200-distilled-600M"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    src_lang="eng_Latn"
)

nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE.type == "cuda" else torch.float32
)

nllb_model = nllb_model.to(DEVICE)
nllb_model.eval()

HINDI_LANG_ID = tokenizer.convert_tokens_to_ids(
    "hin_Deva"
)

print("NLLB loaded successfully.")
print("Hindi language token ID:", HINDI_LANG_ID)

Device: cuda


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

NLLB loaded successfully.
Hindi language token ID: 256068


In [5]:
# ============================================================
# STEP 4: TEST NLLB WITH SIMPLE SENTENCES
# ============================================================

def translate_nllb(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(DEVICE)

    with torch.no_grad():

        generated_tokens = nllb_model.generate(
            **inputs,
            forced_bos_token_id=HINDI_LANG_ID,
            num_beams=4,
            max_new_tokens=128
        )

    translation = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0]

    return translation


test_sentences = [
    "I am happy.",
    "How are you?",
    "Students are studying in the school.",
    "The weather is very good today.",
    "India is a diverse country."
]


for sentence in test_sentences:

    hindi = translate_nllb(sentence)

    print("English:", sentence)
    print("Hindi  :", hindi)
    print("-" * 60)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


English: I am happy.
Hindi  : मैं खुश हूँ।
------------------------------------------------------------
English: How are you?
Hindi  : तुम कैसे हो?
------------------------------------------------------------


[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


English: Students are studying in the school.
Hindi  : छात्र स्कूल में अध्ययन कर रहे हैं।
------------------------------------------------------------
English: The weather is very good today.
Hindi  : आज मौसम बहुत अच्छा है।
------------------------------------------------------------
English: India is a diverse country.
Hindi  : भारत एक विविध देश है।
------------------------------------------------------------


In [6]:
# ============================================================
# NLLB OFFICIAL TEST EVALUATION
# SAME 2507 TEST SENTENCES
# ============================================================

import os
import pandas as pd
import torch
import sacrebleu
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. LOAD TEST DATA
# ------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/NNDL_English_Hindi_Translator"

possible_paths = [
    os.path.join(
        PROJECT_ROOT,
        "data",
        "processed",
        "test_clean.parquet"
    ),
    os.path.join(
        PROJECT_ROOT,
        "processed_data",
        "test_clean.parquet"
    )
]

TEST_PATH = None

for path in possible_paths:
    if os.path.exists(path):
        TEST_PATH = path
        break

if TEST_PATH is None:
    raise FileNotFoundError("test_clean.parquet not found.")

test_df = pd.read_parquet(TEST_PATH)

print("Official test samples:", len(test_df))


# ------------------------------------------------------------
# 2. BATCH TRANSLATION FUNCTION
# ------------------------------------------------------------

def translate_batch_nllb(texts):

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(DEVICE)

    with torch.no_grad():

        generated = nllb_model.generate(
            **inputs,
            forced_bos_token_id=HINDI_LANG_ID,
            num_beams=4,
            max_new_tokens=128
        )

    translations = tokenizer.batch_decode(
        generated,
        skip_special_tokens=True
    )

    return translations


# ------------------------------------------------------------
# 3. TRANSLATE ALL 2507 SENTENCES
# ------------------------------------------------------------

BATCH_SIZE = 8

nllb_predictions = []
nllb_references = test_df["hindi"].astype(str).tolist()

english_sentences = test_df["english"].astype(str).tolist()

for i in tqdm(
    range(0, len(english_sentences), BATCH_SIZE),
    desc="NLLB Test Evaluation"
):

    batch = english_sentences[
        i:i + BATCH_SIZE
    ]

    translations = translate_batch_nllb(batch)

    nllb_predictions.extend(translations)


# ------------------------------------------------------------
# 4. BLEU
# ------------------------------------------------------------

nllb_bleu = sacrebleu.corpus_bleu(
    nllb_predictions,
    [nllb_references]
)


# ------------------------------------------------------------
# 5. chrF
# ------------------------------------------------------------

nllb_chrf = sacrebleu.corpus_chrf(
    nllb_predictions,
    [nllb_references]
)


# ------------------------------------------------------------
# 6. FINAL RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("NLLB OFFICIAL TEST RESULTS")
print("=" * 60)

print("Model : facebook/nllb-200-distilled-600M")
print(f"BLEU  : {nllb_bleu.score:.4f}")
print(f"chrF  : {nllb_chrf.score:.4f}")


# ------------------------------------------------------------
# 7. SHOW 5 EXAMPLES
# ------------------------------------------------------------

print("\nSample translations:\n")

for i in range(5):

    print("English   :", english_sentences[i])
    print("Reference :", nllb_references[i])
    print("NLLB      :", nllb_predictions[i])
    print("-" * 70)

Official test samples: 2507


NLLB Test Evaluation:   0%|          | 0/314 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface


NLLB OFFICIAL TEST RESULTS
Model : facebook/nllb-200-distilled-600M
BLEU  : 23.4315
chrF  : 51.0445

Sample translations:

English   : A black box in your car?
Reference : आपकी कार में ब्लैक बॉक्स?
NLLB      : अपनी कार में एक काला बॉक्स?
----------------------------------------------------------------------
English   : As America's road planners struggle to find the cash to mend a crumbling highway system, many are beginning to see a solution in a little black box that fits neatly by the dashboard of your car.
Reference : जबकि अमेरिका के सड़क योजनाकार, ध्वस्त होते हुए हाईवे सिस्टम को सुधारने के लिए धन की कमी से जूझ रहे हैं, वहीं बहुत-से लोग इसका समाधान छोटे से ब्लैक बॉक्स में देख रहे हैं, जो आपकी कार के डैशबोर्ड पर सफ़ाई से फिट हो जाता है।
NLLB      : जैसे-जैसे अमेरिका के सड़क नियोजकों को एक टूटी हुई राजमार्ग प्रणाली को ठीक करने के लिए नकदी खोजने के लिए संघर्ष करना पड़ता है, कई लोग एक छोटे से ब्लैक बॉक्स में एक समाधान देखना शुरू कर रहे हैं जो आपकी कार के डैशबोर्ड पर अच्छी तरह से फिट ब